# 🧠 Vanilla RNN
**Course:** Deep Learning  
**Lecture:** 5 — Recurrent Neural Networks (RNNs)  
**Part:** a — Vanilla RNN

---

> **Where we are:** Before we build complex sequence models, we need to understand what a single recurrent neuron computes and how sequences are processed.

**What you'll learn:**
- The RNN cell equation and how it differs from a standard feed-forward layer
- Unrolling through time and Backpropagation Through Time (BPTT)
- The vanishing gradient problem in RNNs
- How to implement a vanilla RNN from scratch and using Keras
- When to use `return_sequences=True` vs `return_sequences=False`

**Exam relevance:** ⭐⭐⭐ (Crucial for understanding sequence processing and basic RNN shapes)


In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import os
import pandas as pd

# Confirm device
print(f"TensorFlow {tf.__version__} | Devices: {[d.name for d in tf.config.list_physical_devices()]}")

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12


## 📖 1. The RNN Cell Equation

A Recurrent Neural Network processes sequences by keeping a hidden state $h_t$ that is updated at each timestep.
The equation for a vanilla RNN cell is:
$h_t = \tanh(W_h h_{t-1} + W_x x_t + b)$

Where:
- $x_t$ is the input at timestep $t$
- $h_{t-1}$ is the hidden state from the previous timestep
- $W_x$ and $W_h$ are the weight matrices for the input and hidden state
- $b$ is the bias vector
- $\tanh$ is the activation function that keeps the hidden state values between -1 and 1

**Unrolling through time:** To compute gradients, we unfold the network across all timesteps and apply Backpropagation Through Time (BPTT).

**Vanishing Gradient Problem:** When multiplying gradients through many timesteps (deep network in time), if the weight matrix norms are less than 1, the gradients vanish, making it hard to learn long-range dependencies.


## 🔀 Your Options: Processing Sequences

At this step, you can choose between:

| Option | Pros | Cons | When to use |
|---|---|---|---|
| Numpy RNN | Deeply understand the math | Slow, no autograd | Learning the basics |
| Keras SimpleRNN | Fast, easy to use | Hides the details | Standard vanilla RNN tasks |


In [ ]:
# ✏️ Your Turn: Implement RNN from scratch

# Given dimensions
input_dim = 4
hidden_dim = 3
seq_length = 5

# Initialize random weights and bias
Wx = np.random.randn(input_dim, hidden_dim)
Wh = np.random.randn(hidden_dim, hidden_dim)
b = np.random.randn(hidden_dim)

# Dummy sequence: shape (seq_length, input_dim)
X = np.random.randn(seq_length, input_dim)

# 1. Implement one step of the RNN cell
def rnn_step(x_t, h_prev):
    ### YOUR CODE HERE ###
    # Compute: h_t = tanh(h_prev * Wh + x_t * Wx + b)
    # Note: Pay attention to dot product order
    pass
    
# 2. Implement the full forward pass
def rnn_forward(X):
    # Initialize hidden state h_0 to zeros
    h = np.zeros((1, hidden_dim))
    all_h = []
    
    ### YOUR CODE HERE ###
    # Loop over X, apply rnn_step, append to all_h
    
    return np.array(all_h)


In [ ]:
# ✅ Solution: Implement RNN from scratch

def rnn_step(x_t, h_prev):
    # x_t is (1, input_dim), Wx is (input_dim, hidden_dim) -> (1, hidden_dim)
    # h_prev is (1, hidden_dim), Wh is (hidden_dim, hidden_dim) -> (1, hidden_dim)
    z = np.dot(x_t, Wx) + np.dot(h_prev, Wh) + b
    return np.tanh(z)

def rnn_forward(X):
    h = np.zeros((1, hidden_dim))
    all_h = []
    for t in range(X.shape[0]):
        x_t = X[t:t+1, :]
        h = rnn_step(x_t, h)
        all_h.append(h.flatten())
    return np.array(all_h)

h_states = rnn_forward(X)
print("Hidden states over time:\n", h_states)


## 📖 2. Keras `return_sequences`

When using Keras RNN layers, `return_sequences` is a critical argument.
- `return_sequences=False` (default): Returns only the last hidden state $h_T$. Useful for many-to-one tasks like text classification.
- `return_sequences=True`: Returns the hidden state for ALL timesteps $[h_1, h_2, ..., h_T]$. Useful for many-to-many tasks like sequence translation, or when stacking RNNs.


In [ ]:
# ✏️ Your Turn: Keras SimpleRNN and return_sequences

# We have a batch of sequences: shape (batch_size, sequence_length, features)
X_keras = np.random.randn(32, 10, 4)

### YOUR CODE HERE ###
# 1. Create a tf.keras.layers.SimpleRNN with units=16 and return_sequences=True
# 2. Pass X_keras through it and print the output shape

# 3. Create a tf.keras.layers.SimpleRNN with units=16 and return_sequences=False
# 4. Pass X_keras through it and print the output shape



In [ ]:
# ✅ Solution: Keras SimpleRNN and return_sequences

# 1. return_sequences=True
rnn_seq = tf.keras.layers.SimpleRNN(units=16, return_sequences=True)
out_seq = rnn_seq(X_keras)
print("Shape with return_sequences=True: ", out_seq.shape)
# Output is (batch_size, seq_length, units) -> (32, 10, 16)

# 2. return_sequences=False
rnn_last = tf.keras.layers.SimpleRNN(units=16, return_sequences=False)
out_last = rnn_last(X_keras)
print("Shape with return_sequences=False:", out_last.shape)
# Output is (batch_size, units) -> (32, 16)


In [ ]:
# 🔬 Comparison: Training SimpleRNN and Vanishing Gradient

# We will load the Epileptic Seizure dataset if available, otherwise generate synthetic data.
csv_path = '../05_Practicals/Epileptic_Seizure_Recognition.csv'

if os.path.exists(csv_path):
    print("Loading Epileptic Seizure dataset...")
    df = pd.read_csv(csv_path)
    X_data = df.iloc[:, 1:-1].values
    y_data = (df.iloc[:, -1].values == 1).astype(int)
else:
    print("CSV not found. Generating synthetic dataset...")
    X_data = np.random.randn(1000, 178)
    y_data = np.random.randint(0, 2, size=(1000,))

# We will compare training on seq_length 10, 50, 100 to show accuracy degradation
def create_dataset(seq_len):
    # Truncate to seq_len
    X_trunc = X_data[:, :seq_len]
    X_reshaped = np.expand_dims(X_trunc, -1) # (samples, seq_len, 1)
    return X_reshaped, y_data

seq_lengths = [10, 50, 100]
histories = {}

for seq_len in seq_lengths:
    X_train, y_train = create_dataset(seq_len)
    
    model = tf.keras.Sequential([
        tf.keras.layers.SimpleRNN(32, input_shape=(seq_len, 1)),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    print(f"\nTraining on sequence length: {seq_len}")
    hist = model.fit(X_train, y_train, epochs=5, validation_split=0.2, verbose=0)
    histories[seq_len] = hist.history['val_accuracy']

# Plotting the results
plt.figure(figsize=(10, 6))
for seq_len in seq_lengths:
    plt.plot(histories[seq_len], label=f'Seq Len {seq_len}')
plt.title('Validation Accuracy vs Sequence Length (Vanishing Gradient)')
plt.xlabel('Epochs')
plt.ylabel('Val Accuracy')
plt.legend()
plt.show()

# Memory Visualization
plt.figure(figsize=(10, 4))
plt.imshow(h_states.T, aspect='auto', cmap='viridis')
plt.colorbar()
plt.title('Hidden State Values over Timesteps (Numpy RNN)')
plt.xlabel('Timestep')
plt.ylabel('Hidden Unit')
plt.show()


## 🎓 Exam Corner

### Question 1 (Conceptual)
Write the vanilla RNN cell equation. What is backpropagation through time (BPTT) and what problem does it cause?

<details><summary>💡 Answer</summary>

Equation: $h_t = \tanh(W_h h_{t-1} + W_x x_t + b)$

BPTT is backpropagation applied to the unrolled computational graph of an RNN across all timesteps. Because the same weights ($W_h$) are multiplied repeatedly across timesteps, if their values (and gradients) are small, the gradients shrink exponentially as they propagate backwards (Vanishing Gradient). This makes the RNN struggle to learn long-range dependencies.
</details>

### Question 2 (Practical)
You have 3 stacked RNN layers. Which ones need `return_sequences=True`? Why?

<details><summary>💡 Answer</summary>

The first and second layers need `return_sequences=True`.
Why? A recurrent layer expects a 3D input of shape `(batch, timesteps, features)`. If a lower layer has `return_sequences=False`, it only outputs a 2D tensor `(batch, features)`, which the next RNN layer cannot process. The last layer can be `True` or `False` depending on the final task (e.g., False for classification).
</details>
